# Frequency-dependent selection

How does a copying bias change the distribution of cultural variants?

Student version. Complete the `...` placeholders, then run the checks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(context="paper", 
              style="ticks", 
              font_scale=1.1,
              rc={"axes.spines.top": False, 
                  "axes.spines.right": False})

## 1. Two variants

Variant A has frequency $f$ and variant B has frequency $1-f$. A learner copies A with probability

$$p_A = \frac{f^a}{f^a + (1-f)^a}.$$

Run the curves below. Which values of `a` favour common variants? Which favour rare variants?

In [ ]:
f = np.linspace(0, 1, 200)

for a in [0.5, 1, 2, 5]:
    p = f**a / (f**a + (1 - f)**a)
    plt.plot(f, p, label=f"a = {a}")

plt.plot(f, f, "k--", alpha=0.4, label="Unbiased copying")
plt.xlabel("Frequency of variant A")
plt.ylabel("Probability of copying variant A")
plt.legend(frameon=False)
plt.show()

## 2. n variants

For frequencies $f_1,\ldots,f_n$ that sum to 1,

$$p_i = \frac{f_i^a}{\sum_{j=1}^{n} f_j^a}.$$

Assume the other $n-1$ variants share the remaining frequency equally. Complete `p` below and compare 2, 3, and 10 variants. Then repeat with `a=1` and `a=0.5`. What changes?

In [ ]:
f = np.linspace(0, 1, 500)
a = 2

for n_variants in [2, 3, 10]:
    # Other variants share the remaining frequency equally.
    other_frequency = (1 - f) / (n_variants - 1)
    p = ...  # TODO 1: copying probability of the focal variant
    plt.plot(f, p, label=f"{n_variants} variants")

plt.plot(f, f, "k--", alpha=0.4, label="Unbiased copying")
plt.xlabel("Frequency of focal variant")
plt.ylabel("Probability of copying focal variant")
plt.legend(frameon=False)
plt.show()

Now use frequencies `[0.5, 0.3, 0.2]` and `a=2`. Calculate the copying probabilities: raise the frequencies to `a`, then normalise the weights to sum to 1.

In [ ]:
frequencies = np.array([0.5, 0.3, 0.2])
a = 2
probabilities = ...  # TODO 2: turn frequencies into copying weights
probabilities = ...  # TODO 3: normalise the weights to sum to 1

print("Copying probabilities:", probabilities.round(3))
assert np.all(probabilities >= 0) and np.isclose(probabilities.sum(), 1)

## 3. Build the model

Each agent carries one variant. Every generation, count variants, copy using their probabilities, then let each learner invent a unique variant with probability `mutation_rate`.

Complete the three missing steps below:

1. Convert counts to frequencies.
2. Draw the next population with `np.random.choice`.
3. Draw the innovation mask using `np.random.random(n_agents)` and `mutation_rate`.

The new-variant labels are handled for you. The function returns final frequencies, sorted from largest to smallest.

In [ ]:
def frequency_bias(n_agents=1000, n_variants=100,
                   n_generations=100, a=2, mutation_rate=0.05):
    agents = np.random.randint(n_variants, size=n_agents)
    next_variant = n_variants

    for _ in range(n_generations):
        # 1. Count the variants in the current generation.
        variants, counts = np.unique(agents, return_counts=True)
        frequencies = ...  # TODO 4: convert counts to frequencies

        # 2. Apply the same copying rule, then sample the next generation.
        probabilities = frequencies**a
        probabilities /= probabilities.sum()
        agents = ...  # TODO 5: sample n_agents variants using probabilities

        # 3. Give each innovator a unique new variant.
        mutations = ...  # TODO 6: draw one innovation decision per learner
        n_new = mutations.sum()
        agents[mutations] = np.arange(next_variant, next_variant + n_new)
        next_variant += n_new

    _, counts = np.unique(agents, return_counts=True)
    return np.sort(counts / n_agents)[::-1]

In [ ]:
# Final frequencies must be positive, sum to 1, and be sorted by rank.
np.random.seed(7)
check = frequency_bias(n_agents=50, n_variants=3, n_generations=5,
                       mutation_rate=0)
assert np.isclose(check.sum(), 1) and np.all(check > 0)
assert np.all(np.diff(check) <= 0) and len(check) <= 3
check = frequency_bias(n_agents=50, n_variants=1, n_generations=5,
                       mutation_rate=0)
assert np.array_equal(check, [1.0])

## 4. Compare copying rules

Run ten populations for each rule and plot their final frequencies by rank. Each line is one run; rank 1 is the most common variant.

Which rule produces the most dominant variant? Which gives the most even distribution?

In [ ]:
np.random.seed(7)
n_runs = 10
rank_frequencies_drift = [frequency_bias(a=1) for _ in range(n_runs)]
rank_frequencies_PFDS = [frequency_bias(a=1.1) for _ in range(n_runs)]
rank_frequencies_NFDS = [frequency_bias(a=0.5) for _ in range(n_runs)]

In [ ]:
plt.figure(figsize=(8, 4))
for runs, color, label in [
    (rank_frequencies_drift, "0.4", "Unbiased copying: a = 1"),
    (rank_frequencies_PFDS, "#0072B2", "Positive frequency dependence: a = 1.1"),
    (rank_frequencies_NFDS, "#D55E00", "Negative frequency dependence: a = 0.5"),
]:
    for run, frequencies in enumerate(runs):
        plt.plot(np.arange(1, len(frequencies) + 1), frequencies,
                 color=color, alpha=0.3, label=label if run == 0 else None)

plt.xscale("log")
plt.yscale("log")
plt.xlabel("Variant rank (1 = most common)")
plt.ylabel("Final frequency")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

## 5. Innovation

Keep `a=0.5`. Compare `mutation_rate=0` and `0.05`, with ten runs each. Plot the rank-frequency curves, keeping all other parameters fixed.

How does innovation change the number of rare variants? Can frequency-dependence alone keep a low frequency variant from dissapearing?

In [ ]:
# Compare innovation rates 0 and 0.05 with a=0.5.
# YOUR CODE HERE